In [1]:
import stim

In [2]:
import torch
import torch.nn.functional as F


In [3]:
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.utils import add_self_loops, degree


In [4]:
import matplotlib.pyplot as plt


In [5]:
import numpy as np
from tqdm.auto import trange
import networkx as nx
import math
from sklearn.metrics import roc_auc_score

c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# ============================================================================
# PART 1: SURFACE CODE CIRCUIT
# ============================================================================

def surface_code_circuit(p, d):
    """Generate surface code circuit with given error rate and distance"""
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=d,
        distance=d,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p
    )

In [8]:
class SyndromeGraph:
    """
    Pre-built graph structure with SEPARATED spatial and temporal edges.
    Spatial: bidirectional (undirected)
    Temporal: unidirectional (t → t+1 only)
    """

    def __init__(self, circuit, device):
        self.device = device
        self.num_nodes = circuit.num_detectors

        # Build SEPARATE edge sets
        self.spatial_edge_index, self.temporal_edge_index = self._build_edges(circuit)
        self.spatial_edge_index = self.spatial_edge_index.to(device)
        self.temporal_edge_index = self.temporal_edge_index.to(device)

        self.num_spatial_edges = self.spatial_edge_index.shape[1]
        self.num_temporal_edges = self.temporal_edge_index.shape[1]

        print(f"✓ Graph skeleton built: {self.num_nodes} detectors")
        print(f"  Spatial edges (undirected): {self.num_spatial_edges}")
        print(f"  Temporal edges (directed): {self.num_temporal_edges}")

    def _build_edges(self, circuit):
        """
        Build TWO separate edge sets:
        1. Spatial: bidirectional, same time, adjacent in space
        2. Temporal: unidirectional, same space, forward in time (t → t+1)
        """
        detector_coords = circuit.get_detector_coordinates()
        spatial_edges = []
        temporal_edges = []

        # Build lookup: (x, y, t) → detector_id
        coord_to_id = {}
        for det_id, coord in detector_coords.items():
            key = (coord[0], coord[1], coord[2])
            coord_to_id[key] = det_id

        for i in range(self.num_nodes):
            coord_i = detector_coords.get(i, [0, 0, 0])
            xi, yi, ti = coord_i[0], coord_i[1], coord_i[2]

            # SPATIAL EDGES: same time, adjacent in space
            for j in range(i + 1, self.num_nodes):
                coord_j = detector_coords.get(j, [0, 0, 0])
                xj, yj, tj = coord_j[0], coord_j[1], coord_j[2]

                if ti == tj:  # Same time round
                    dx = abs(xi - xj)
                    dy = abs(yi - yj)

                    # Adjacent means differ by 2 in exactly one coordinate
                    if (dx == 2 and dy == 0) or (dx == 0 and dy == 2):
                        spatial_edges.append([i, j])
                        spatial_edges.append([j, i])  # Bidirectional

            # TEMPORAL EDGES: same position, forward in time (t → t+1)
            next_key = (xi, yi, ti + 1)
            if next_key in coord_to_id:
                j = coord_to_id[next_key]
                temporal_edges.append([i, j])  # ONLY FORWARD (unidirectional)

        # Fallback if no edges found
        if len(spatial_edges) == 0:
            print("  Warning: No spatial edges, using fallback")
            for i in range(self.num_nodes - 1):
                spatial_edges.append([i, i + 1])
                spatial_edges.append([i + 1, i])

        if len(temporal_edges) == 0:
            print("  Warning: No temporal edges")

        spatial_tensor = torch.tensor(spatial_edges, dtype=torch.long).t().contiguous() if spatial_edges else torch.empty((2, 0), dtype=torch.long)
        temporal_tensor = torch.tensor(temporal_edges, dtype=torch.long).t().contiguous() if temporal_edges else torch.empty((2, 0), dtype=torch.long)

        return spatial_tensor, temporal_tensor

    def create_batch(self, syndrome_values):
        """
        Create batched graph data with SEPARATE spatial and temporal edges.
        """
        batch_size = syndrome_values.shape[0]

        # Node features: (batch_size * num_nodes, 1)
        x = syndrome_values[:, :self.num_nodes].reshape(-1, 1).float()

        # Replicate spatial edges for each graph in batch
        spatial_edge_indices = []
        for i in range(batch_size):
            offset = i * self.num_nodes
            spatial_edge_indices.append(self.spatial_edge_index + offset)
        spatial_edge_index = torch.cat(spatial_edge_indices, dim=1) if spatial_edge_indices else torch.empty((2, 0), dtype=torch.long, device=self.device)

        # Replicate temporal edges for each graph in batch
        temporal_edge_indices = []
        for i in range(batch_size):
            offset = i * self.num_nodes
            temporal_edge_indices.append(self.temporal_edge_index + offset)
        temporal_edge_index = torch.cat(temporal_edge_indices, dim=1) if temporal_edge_indices else torch.empty((2, 0), dtype=torch.long, device=self.device)

        # Batch assignment
        batch = torch.arange(batch_size, device=self.device).repeat_interleave(self.num_nodes)

        return Data(
            x=x,
            spatial_edge_index=spatial_edge_index,
            temporal_edge_index=temporal_edge_index,
            batch=batch
        )

In [ ]:
class ChebConvLayer(torch.nn.Module):
    """
    Chebyshev Polynomial Graph Convolution.

    Computes: h_out = Σ_{k=0}^K θ_k · T_k(L̃) · h_in

    Where:
    - T_k are Chebyshev polynomials
    - L̃ is the normalized graph Laplacian
    - θ_k are learnable coefficients

    Supports both DIRECTED (random-walk) and UNDIRECTED (symmetric) Laplacians.
    """

    def __init__(self, in_channels, out_channels, K, normalization='rw'):
        """
        Args:
            in_channels: Input feature dimension
            out_channels: Output feature dimension
            K: Order of Chebyshev polynomial (degree)
            normalization: 'rw' (random-walk) for directed, 'sym' (symmetric) for undirected
        """
        super().__init__()
        self.K = K
        self.normalization = normalization

        # Linear transformation for concatenated polynomial outputs
        self.lin = torch.nn.Linear(in_channels * (K + 1), out_channels, bias=False)
        self.bias = torch.nn.Parameter(torch.zeros(out_channels))

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        """
        Args:
            x: Node features (num_nodes, in_channels)
            edge_index: Edge connectivity (2, num_edges)

        Returns:
            h: Transformed features (num_nodes, out_channels)
        """
        num_nodes = x.size(0)

        # Compute normalized Laplacian (FIXED: pass x.dtype)
        L_norm = self._compute_laplacian(edge_index, num_nodes, x.dtype)

        # Compute Chebyshev polynomials T_0, T_1, ..., T_K
        Tx_list = []

        # T_0(L) = I, so T_0(L)·x = x
        Tx_list.append(x)

        if self.K >= 1:
            # T_1(L) = L, so T_1(L)·x = L·x
            Tx_1 = self._sparse_mm(L_norm, x, num_nodes)
            Tx_list.append(Tx_1)

        # Recursive: T_k(L) = 2·L·T_{k-1}(L) - T_{k-2}(L)
        for k in range(2, self.K + 1):
            Tx_k = 2.0 * self._sparse_mm(L_norm, Tx_list[-1], num_nodes) - Tx_list[-2]
            Tx_list.append(Tx_k)

        # Concatenate all polynomial orders: (num_nodes, in_channels * (K+1))
        Tx_cat = torch.cat(Tx_list, dim=1)

        # Linear transformation
        out = self.lin(Tx_cat) + self.bias

        return out

    def _compute_laplacian(self, edge_index, num_nodes, dtype):
        """
        Compute normalized Laplacian based on edge directionality.

        For DIRECTED graphs (temporal): L = I - D^{-1}A (random walk)
        For UNDIRECTED graphs (spatial): L = I - D^{-1/2}AD^{-1/2} (symmetric)

        Returns edge_index and edge_weight for sparse matrix operations.
        """
        row, col = edge_index[0], edge_index[1]

        # Compute degree (FIXED: use dtype parameter)
        deg = degree(row, num_nodes, dtype=dtype)

        if self.normalization == 'rw':
            # Random walk: D^{-1}A
            deg_inv = deg.pow(-1.0)
            deg_inv[deg_inv == float('inf')] = 0
            edge_weight = deg_inv[row]

        elif self.normalization == 'sym':
            # Symmetric: D^{-1/2}AD^{-1/2}
            deg_inv_sqrt = deg.pow(-0.5)
            deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
            edge_weight = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        else:
            raise ValueError(f"Unknown normalization: {self.normalization}")

        # L = I - normalized_A
        # We'll compute this implicitly in _sparse_mm
        return (edge_index, edge_weight)

    def _sparse_mm(self, laplacian_data, x, num_nodes):
        """
        Compute L·x efficiently using sparse operations.

        Since L = I - A_norm, we compute:
        L·x = x - A_norm·x
        """
        edge_index, edge_weight = laplacian_data

        # Compute A_norm · x (message passing with weights)
        out = torch.zeros_like(x)
        row, col = edge_index[0], edge_index[1]

        # Efficient scatter: out[row] += edge_weight * x[col]
        for i in range(x.size(1)):  # For each feature channel
            out[:, i].scatter_add_(0, row, edge_weight * x[col, i])

        # L·x = x - A_norm·x
        return x - out

In [10]:
class HybridPolyConv(torch.nn.Module):
    """
    Hybrid polynomial filter with SEPARATE spatial and temporal processing.

    Architecture:
    - Spatial polynomial (undirected, symmetric Laplacian)
    - Temporal polynomial (directed, random-walk Laplacian)
    - Combine outputs

    This preserves the distinct nature of space and time while allowing
    multi-hop information flow in each domain.
    """

    def __init__(self, in_channels, out_channels, K=5):
        super().__init__()

        # Spatial: undirected, symmetric normalization
        self.spatial_conv = ChebConvLayer(
            in_channels,
            out_channels // 2,
            K=K,
            normalization='sym'  # Symmetric for undirected
        )

        # Temporal: directed, random-walk normalization
        self.temporal_conv = ChebConvLayer(
            in_channels,
            out_channels // 2,
            K=K,
            normalization='rw'  # Random-walk for directed
        )

        # Combine spatial and temporal features
        self.combine = torch.nn.Linear(out_channels, out_channels)

    def reset_parameters(self):
        self.spatial_conv.reset_parameters()
        self.temporal_conv.reset_parameters()
        self.combine.reset_parameters()

    def forward(self, x, spatial_edge_index, temporal_edge_index):
        """
        Args:
            x: Node features (num_nodes, in_channels)
            spatial_edge_index: Spatial edges (bidirectional)
            temporal_edge_index: Temporal edges (unidirectional t→t+1)

        Returns:
            Combined spatial-temporal features
        """
        # Process spatial relationships (undirected)
        h_spatial = self.spatial_conv(x, spatial_edge_index)

        # Process temporal relationships (directed, forward only)
        h_temporal = self.temporal_conv(x, temporal_edge_index)

        # Concatenate and combine
        h_combined = torch.cat([h_spatial, h_temporal], dim=1)

        return self.combine(h_combined)

In [11]:
class SyndromeDecoder(torch.nn.Module):
    """
    GNN decoder using Polynomial Graph Filters.

    Architecture:
    - Multiple HybridPolyConv layers (spatial + temporal polynomials)
    - Layer normalization after each layer
    - Global pooling
    - MLP head for classification
    """

    def __init__(self, hidden_dim=128, num_layers=4, K=5):
        """
        Args:
            hidden_dim: Hidden feature dimension
            num_layers: Number of polynomial conv layers
            K: Polynomial order (how many hops to consider)
        """
        super().__init__()

        self.K = K

        # Polynomial convolution layers
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        # First layer: 1 input (syndrome value) → hidden
        self.layers.append(HybridPolyConv(1, hidden_dim, K=K))
        self.norms.append(torch.nn.LayerNorm(hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 1):
            self.layers.append(HybridPolyConv(hidden_dim, hidden_dim, K=K))
            self.norms.append(torch.nn.LayerNorm(hidden_dim))

        # Output head
        self.output = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 64),
            torch.nn.SiLU(),
            torch.nn.Linear(64, 1)
        )

        print(f"✓ Polynomial GNN built:")
        print(f"  - Polynomial order K={K}")
        print(f"  - Hidden dimension: {hidden_dim}")
        print(f"  - Number of layers: {num_layers}")

    def forward(self, data):
        """
        Args:
            data: PyG Data object with:
                - x: node features
                - spatial_edge_index: spatial edges
                - temporal_edge_index: temporal edges
                - batch: batch assignment

        Returns:
            logits: (batch_size, 1) prediction logits
        """
        x = data.x
        spatial_edges = data.spatial_edge_index
        temporal_edges = data.temporal_edge_index
        batch = data.batch

        # Multi-hop message passing with polynomial filters
        for layer, norm in zip(self.layers, self.norms):
            x = layer(x, spatial_edges, temporal_edges)
            x = norm(x)
            x = F.silu(x)

        # Global pooling: aggregate all nodes in each graph
        graph_features = global_mean_pool(x, batch)

        # Predict logical error
        return self.output(graph_features)

In [16]:
class GNNTrainer:
    """Clean interface for training polynomial GNN decoder with progress tracking"""

    def __init__(self, p, d, device, hidden_dim=128, num_layers=4, K=5):
        self.p = p
        self.d = d
        self.device = device

        print(f"\n{'='*60}")
        print(f"Initializing Polynomial GNN Trainer")
        print(f"  Code: d={d}, p={p}")
        print(f"  Polynomial order: K={K}")
        print(f"{'='*60}")

        # Build circuit (ONCE)
        print("\n[1/3] Building quantum circuit...")
        self.circuit = surface_code_circuit(p, d)

        # Build graph structure with SEPARATED edges (ONCE)
        print("\n[2/3] Building graph skeleton...")
        self.graph = SyndromeGraph(self.circuit, device)

        # Build polynomial model
        print("\n[3/3] Building polynomial neural network...")
        self.model = SyndromeDecoder(
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            K=K  # Polynomial order
        ).to(device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)

        # Count parameters
        num_params = sum(p.numel() for p in self.model.parameters())
        print(f"\n✓ Total parameters: {num_params:,}")
        print(f"\n{'='*60}")
        print("Ready to train!")
        print(f"{'='*60}\n")

    def sample_data(self, num_samples):
        """Generate syndrome samples"""
        sampler = self.circuit.compile_detector_sampler()
        detections, flips = sampler.sample(num_samples, separate_observables=True)

        # Convert: 0/1 -> -1/+1
        # Use .copy() to ensure contiguous array
        syndrome_np = (detections.astype(np.int64) * 2 - 1).copy()
        labels_np = flips.astype(np.int64).flatten().copy()

        syndromes = torch.from_numpy(syndrome_np).to(self.device)
        labels = torch.from_numpy(labels_np).to(self.device)

        return syndromes, labels

    def train_epoch(self, syndromes, labels, batch_size=256):
        """One training epoch with progress bar"""
        self.model.train()
        num_samples = syndromes.shape[0]
        num_batches = math.ceil(num_samples / batch_size)

        # Shuffle data at the start of each epoch
        perm = torch.randperm(num_samples, device=self.device)
        syndromes = syndromes[perm]
        labels = labels[perm]

        # Compute class weights for imbalanced data
        eps = 1e-6
        num_pos = labels.sum().item()
        num_neg = num_samples - num_pos
        pos_weight = torch.tensor([(num_neg + eps) / (num_pos + eps)], device=self.device)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        total_loss = 0
        correct = 0
        samples_seen = 0

        with trange(num_batches, desc="Training", leave=False) as pbar:
            for batch_idx in pbar:
                start = batch_idx * batch_size
                end = min(start + batch_size, num_samples)

                batch_syn = syndromes[start:end]
                batch_lab = labels[start:end]
                batch_len = end - start

                # Create graph batch
                batch_data = self.graph.create_batch(batch_syn)

                # Forward
                logits = self.model(batch_data)
                loss = loss_fn(logits, batch_lab.unsqueeze(1).float())

                # Backward
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                # Track metrics
                total_loss += loss.item()
                pred = torch.sigmoid(logits)
                batch_correct = ((pred > 0.5).flatten() == batch_lab).sum().item()
                correct += batch_correct
                samples_seen += batch_len

                # Update progress bar
                running_acc = correct / samples_seen
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{running_acc:.4f}'
                })

        avg_loss = total_loss / num_batches
        accuracy = correct / num_samples
        return avg_loss, accuracy

    def evaluate(self, num_samples=10000, batch_size=500):
        """Evaluate on fresh data with multiple metrics"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_samples)

        all_probs = []
        with torch.no_grad():
            for i in range(0, num_samples, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                logits = self.model(batch_data)
                prob = torch.sigmoid(logits)
                all_probs.append(prob.flatten().cpu().numpy())

        all_probs = np.concatenate(all_probs)
        labels_np = labels.cpu().numpy()

        # Compute metrics
        all_preds = (all_probs > 0.5).astype(int)
        accuracy = (all_preds == labels_np).mean()

        # AUROC - only compute if both classes present
        if len(np.unique(labels_np)) > 1:
            auroc = roc_auc_score(labels_np, all_probs)
        else:
            auroc = float('nan')

        return accuracy, auroc

    def get_ler(self, num_shots=100000, batch_size=1000):
        """Compute logical error rate"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_shots)

        all_probs = []
        with torch.no_grad():
            for i in range(0, num_shots, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                logits = self.model(batch_data)
                prob = torch.sigmoid(logits)
                all_probs.append(prob.flatten().cpu().numpy())

        all_probs = np.concatenate(all_probs)
        labels_np = labels.cpu().numpy()

        # Use 0.5 threshold
        all_preds = (all_probs > 0.5).astype(int)
        errors = (all_preds != labels_np)
        return errors.mean()

    def save(self, path):
        """Save model weights"""
        torch.save(self.model.state_dict(), path)
        print(f"✓ Model saved to {path}")

    def load(self, path):
        """Load model weights"""
        self.model.load_state_dict(torch.load(path, weights_only=True))
        print(f"✓ Model loaded from {path}")

In [13]:
# ============================================================================
# PART 6: MWPM BASELINE FOR COMPARISON
# ============================================================================

def get_mwpm_accuracy(p, d, num_shots=100000):
    """Compute MWPM accuracy for comparison"""
    import pymatching

    print(f"Computing MWPM baseline ({num_shots:,} shots)...")

    circuit = surface_code_circuit(p, d)
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    predictions = matcher.decode_batch(detection_events)

    num_errors = sum(
        1 for shot in range(num_shots)
        if not np.array_equal(observable_flips[shot], predictions[shot])
    )

    ler = num_errors / num_shots
    accuracy = 1 - ler
    print(f"✓ MWPM Accuracy: {accuracy:.6f} (LER: {ler:.6f})")

    return accuracy

In [14]:
# ============================================================================
# PART 7: PROGRESSIVE TRAINING - Train until beating MWPM
# ============================================================================

def train_until_beat_mwpm(p, d, device, max_train_size=10**8, chunk_size=10**7):
    """Train with progressively larger datasets until beating MWPM"""
    import gc

    print(f"\n{'#'*60}")
    print(f"# PROGRESSIVE TRAINING: d={d}, p={p}")
    print(f"{'#'*60}")

    # Get MWPM baseline first
    print("\n" + "="*60)
    print("STEP 1: Getting MWPM baseline")
    print("="*60)
    mwpm_accuracy = get_mwpm_accuracy(p, d)
    print(f"\n🎯 Target to beat: {mwpm_accuracy:.6f}")

    # Initialize trainer
    print("\n" + "="*60)
    print("STEP 2: Setting up GNN")
    print("="*60)
    trainer = GNNTrainer(p, d, device)

    # Progressive training
    print("\n" + "="*60)
    print("STEP 3: Progressive training")
    print("="*60)

    train_size = 100
    beat_mwpm = False

    while train_size <= max_train_size and not beat_mwpm:
        print(f"\n{'─'*60}")
        print(f"📊 Training with {train_size:,} samples")
        print(f"{'─'*60}")

        # Reset model for fresh start
        trainer.model = SyndromeDecoder(hidden_dim=128, num_layers=4).to(device)
        trainer.optimizer = torch.optim.Adam(trainer.model.parameters(), lr=1e-3)

        # Calculate chunks needed
        num_chunks = max(1, train_size // chunk_size)
        samples_per_chunk = min(train_size, chunk_size)

        # Train in chunks
        for chunk_idx in range(num_chunks):
            current_chunk_size = min(samples_per_chunk, train_size - chunk_idx * chunk_size)

            print(f"\n  Chunk {chunk_idx + 1}/{num_chunks}: {current_chunk_size:,} samples")
            print(f"  Generating data...")

            syndromes, labels = trainer.sample_data(current_chunk_size)

            # Multiple epochs per chunk for small datasets
            num_epochs = max(1, min(10, 10000 // (current_chunk_size // 256 + 1)))

            for epoch in range(num_epochs):
                loss, acc = trainer.train_epoch(syndromes, labels, batch_size=256)
                print(f"    Epoch {epoch+1}/{num_epochs}: Loss={loss:.4f}, Train Acc={acc:.4f}")

            # Cleanup
            del syndromes, labels
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Evaluate
        print(f"\n  Evaluating on fresh test data...")
        gnn_accuracy, gnn_auroc = trainer.evaluate(num_samples=10000)

        print(f"\n  📈 Results:")
        print(f"     GNN Accuracy:  {gnn_accuracy:.6f}")
        print(f"     GNN AUROC:     {gnn_auroc:.6f}")
        print(f"     MWPM Accuracy: {mwpm_accuracy:.6f}")
        print(f"     Difference:    {(gnn_accuracy - mwpm_accuracy)*100:+.3f}%")

        if gnn_accuracy > mwpm_accuracy:
            print(f"\n  🎉 SUCCESS! GNN beats MWPM at train_size = {train_size:,}")
            beat_mwpm = True
        else:
            print(f"\n  ❌ Not yet. Trying 10x more data...")
            train_size *= 10
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not beat_mwpm:
        print(f"\n❌ Did not beat MWPM within max train_size = {max_train_size:,}")
        return None, None

    return trainer, train_size

In [17]:
import os

# Configuration
train_p = 0.005
max_train_size = 10**8
chunk_size = 10**7

results = {}
trained_models = {}

# Train for each distance
for d in [3, 5, 7]:
    model_path = f"gnn_poly_decoder_d{d}_p{train_p}_K5.pt"  # NEW: different filename

    print(f"\n{'█'*60}")
    print(f"█  DISTANCE {d} - POLYNOMIAL FILTERS")
    print(f"{'█'*60}")

    # Check if model already exists
    if os.path.exists(model_path):
        print(f"\n✓ Found existing model: {model_path}")
        trainer = GNNTrainer(train_p, d, device, K=5)  # NEW: specify K
        trainer.load(model_path)
        train_size = None
    else:
        print(f"\n→ No saved model found. Training new model...")
        trainer, train_size = train_until_beat_mwpm(
            p=train_p, d=d, device=device,
            max_train_size=max_train_size, chunk_size=chunk_size
        )

        if trainer is not None:
            trainer.save(model_path)
        else:
            print(f"Training failed for d={d}")
            continue

    trained_models[d] = trainer
    results[d] = train_size

# Print summary
print(f"\n\n{'═'*60}")
print("POLYNOMIAL FILTER RESULTS")
print(f"{'═'*60}")
for d, train_size in results.items():
    if train_size:
        print(f"  Distance {d}: Beat MWPM with {train_size:,} samples")
    else:
        print(f"  Distance {d}: Loaded from saved model")
print(f"{'═'*60}")


████████████████████████████████████████████████████████████
█  DISTANCE 3 - POLYNOMIAL FILTERS
████████████████████████████████████████████████████████████

→ No saved model found. Training new model...

############################################################
# PROGRESSIVE TRAINING: d=3, p=0.005
############################################################

STEP 1: Getting MWPM baseline
Computing MWPM baseline (100,000 shots)...
✓ MWPM Accuracy: 0.982600 (LER: 0.017400)

🎯 Target to beat: 0.982600

STEP 2: Setting up GNN

Initializing Polynomial GNN Trainer
  Code: d=3, p=0.005
  Polynomial order: K=5

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors
  Spatial edges (undirected): 32
  Temporal edges (directed): 16

[3/3] Building polynomial neural network...
✓ Polynomial GNN built:
  - Polynomial order K=5
  - Hidden dimension: 128
  - Number of layers: 4

✓ Total parameters: 371,585

Ready to train!


STEP 3: Progressive tr

NameError: name 'x' is not defined

In [ ]:
# Quick test
print("🧪 Testing Polynomial Filters...")
test_trainer = GNNTrainer(p=0.005, d=3, device=device, K=5)

# Generate data
syndromes, labels = test_trainer.sample_data(1000)

# Train 1 epoch
loss, acc = test_trainer.train_epoch(syndromes, labels)
print(f"✓ Training works! Loss={loss:.4f}, Acc={acc:.4f}")

# Evaluate
test_acc, test_auroc = test_trainer.evaluate(1000)
print(f"✓ Evaluation works! Acc={test_acc:.4f}, AUROC={test_auroc:.4f}")

🧪 Quick Test: Training a small GNN on d=3

Initializing GNN Trainer for d=3, p=0.005

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors, 48 connections
  Average neighbors per detector: 2.0

[3/3] Building neural network...
✓ Model built: 59,137 parameters

Ready to train!


Generating 10,000 training samples...
✓ Syndromes shape: torch.Size([10000, 24])
✓ Labels shape: torch.Size([10000])
✓ Example syndrome: [-1, -1, -1, -1, -1] ...

Training for 3 epochs...


  Epoch 1: Loss=0.4129, Accuracy=0.8907


  Epoch 2: Loss=0.2895, Accuracy=0.8960


  Epoch 3: Loss=0.2861, Accuracy=0.8960

Evaluating on fresh test data...
✓ Test Accuracy: 0.9024

✅ Quick test passed! The GNN is working.
